# Notebook 06_05: Structured Output & Function Calling

**Learning Objectives:**
- Extract structured data (JSON, lists, key-value pairs) from LLM outputs
- Use JSON mode in Ollama for reliable structured generation
- Parse and validate LLM output with Pydantic schemas
- Implement function calling / tool use patterns with local LLMs
- Build a schema-validated extraction pipeline

**Prerequisites:** Notebook 06_01 (MCP Basics), Ollama installed with models pulled

## Prerequisites

### Hardware Requirements

| Requirement | Small (CPU) | Large (GPU) |
|-------------|-------------|-------------|
| **LLM** | llama3.2:1b | llama3.1:8b |
| **LLM Size** | 1.3 GB | 4.7 GB |
| **RAM** | 8 GB minimum | 12 GB |
| **VRAM** | N/A (CPU) | 8 GB+ |

### Software Requirements

```bash
# Ollama (install from ollama.com)
ollama pull llama3.2:1b       # CPU-friendly
ollama pull llama3.1:8b       # GPU-optimized (optional)

# Python libraries
pip install ollama pydantic
```

### Optional Installation

```bash
pip install instructor          # Pydantic-validated LLM outputs
```

## Expected Behaviors

### JSON Mode
- Ollama returns valid JSON when `format="json"` is specified
- Response times: 1-3 seconds for small model, 2-5 seconds for large
- JSON structure follows the schema described in the prompt (but is not enforced by the model)

### Pydantic Validation
- Valid responses parse successfully into Pydantic models
- Invalid responses raise `ValidationError` with clear field-level messages
- Retry logic can recover from occasional malformed outputs

### Function Calling
- The LLM selects which function to call based on the user query
- Function arguments are extracted as structured JSON
- Results are passed back to the LLM for a natural language response

### Common Issues
- **Ollama not running**: Start with `ollama serve` in a terminal
- **Model not pulled**: Run `ollama pull llama3.2:1b` first
- **Malformed JSON**: Smaller models occasionally produce invalid JSON — we handle this with retry logic
- **Wrong schema**: The model may not follow complex schemas perfectly — keep schemas simple

## Overview

LLMs generate free-form text by default, but many applications need **structured output**: JSON for APIs, typed objects for code, or specific formats for downstream processing. This notebook covers three approaches:

| Approach | Reliability | Complexity | Best For |
|----------|------------|------------|----------|
| **Prompt Engineering** | Medium | Low | Simple extractions |
| **JSON Mode** | High | Low | Structured API responses |
| **Pydantic Validation** | Very High | Medium | Type-safe applications |
| **Function Calling** | Very High | Medium-High | Tool-using agents |

### Why Structured Output Matters

- **API Integration**: Downstream services expect JSON, not prose
- **Database Storage**: Extracted entities need typed fields
- **Agentic Workflows**: Tool selection requires structured function calls
- **Reliability**: Validation catches errors before they propagate

## Setup and Installation

In [ ]:
import json
import random
import sys
import time
import warnings
from typing import Any

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# Check Ollama
try:
    import ollama
    # Test connection
    ollama.list()
    OLLAMA_AVAILABLE = True
    print("Ollama: connected")
except ImportError:
    OLLAMA_AVAILABLE = False
    print("Ollama not installed. Install with: pip install ollama")
except Exception:
    OLLAMA_AVAILABLE = False
    print("Ollama not running. Start with: ollama serve")

# Check Pydantic
try:
    from pydantic import BaseModel, Field, ValidationError
    PYDANTIC_AVAILABLE = True
    print(f"Pydantic: available")
except ImportError:
    PYDANTIC_AVAILABLE = False
    print("Pydantic not installed. Install with: pip install pydantic")

# Check instructor (optional)
try:
    import instructor
    INSTRUCTOR_AVAILABLE = True
    print(f"Instructor: available")
except ImportError:
    INSTRUCTOR_AVAILABLE = False
    print("Instructor not available (optional). Install with: pip install instructor")

# Reproducibility
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)

# Version metadata
print(f"\nPython: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Model Selection

In [ ]:
# Option 1: Small model (CPU-friendly, 1.3 GB)
LLM_MODEL = "llama3.2:1b"

# Option 2: Large model (GPU-optimized, 4.7 GB — better at following schemas)
# LLM_MODEL = "llama3.1:8b"

print(f"Using model: {LLM_MODEL}")

if OLLAMA_AVAILABLE:
    # Verify model is available
    available_models = [m.model for m in ollama.list().models]
    # Strip tag suffixes for matching
    model_base = LLM_MODEL.split(":")[0]
    found = any(model_base in m for m in available_models)
    if found:
        print(f"Model '{LLM_MODEL}' is available locally.")
    else:
        print(f"Model '{LLM_MODEL}' not found. Pull with: ollama pull {LLM_MODEL}")
        print(f"Available models: {available_models}")

## Part 1: Prompt Engineering for Structured Output

The simplest approach to getting structured output is careful prompt engineering. We instruct the model to respond in a specific format and parse the result. This works for simple cases but is fragile — the model may not always comply perfectly.

In [ ]:
def query_llm(
    prompt: str,
    model: str = LLM_MODEL,
    json_mode: bool = False,
    temperature: float = 0.0,
) -> str:
    """Send a prompt to Ollama and return the response text.

    Args:
        prompt: The user prompt.
        model: Ollama model name.
        json_mode: Whether to enable JSON output mode.
        temperature: Sampling temperature (0 = deterministic).

    Returns:
        The model's response text.
    """
    if not OLLAMA_AVAILABLE:
        return '{"error": "Ollama not available"}'

    kwargs: dict[str, Any] = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "options": {"temperature": temperature, "seed": SEED},
    }
    if json_mode:
        kwargs["format"] = "json"

    response = ollama.chat(**kwargs)
    return response["message"]["content"]


# Test basic prompt engineering
prompt = """Extract the following information from the text below as JSON:
- name (string)
- age (integer)
- occupation (string)
- skills (list of strings)

Text: "John Smith is a 34-year-old software engineer who specializes in Python, machine learning, and cloud architecture."

Respond ONLY with valid JSON, no other text."""

print("=== PROMPT ENGINEERING APPROACH ===\n")
response = query_llm(prompt)
print(f"Raw response:\n{response}\n")

try:
    parsed = json.loads(response)
    print(f"Parsed successfully:")
    for key, value in parsed.items():
        print(f"  {key}: {value}")
except json.JSONDecodeError as e:
    print(f"JSON parsing failed: {e}")
    print("This is why we need more robust approaches!")

## Part 2: Ollama JSON Mode

Ollama supports a `format="json"` parameter that constrains the model to produce valid JSON. This is more reliable than prompt engineering alone because the model's token generation is guided to produce syntactically correct JSON.

Note that JSON mode guarantees *syntactic* validity (parseable JSON) but not *semantic* validity (correct fields and types). We still need the prompt to describe the desired schema.

In [ ]:
# JSON mode extraction
prompt = """Extract person information from this text as JSON with keys:
"name", "age", "occupation", "skills", "location"

Text: "Maria Garcia, age 28, works as a data scientist in Barcelona. She is skilled in R, statistics, deep learning, and data visualization."

Return JSON only."""

print("=== JSON MODE ===\n")
response = query_llm(prompt, json_mode=True)
print(f"Response:\n{response}\n")

parsed = json.loads(response)
print("Parsed fields:")
for key, value in parsed.items():
    print(f"  {key:12s} : {value}")

In [ ]:
# Multi-entity extraction with JSON mode
prompt = """Extract ALL people mentioned in the text as a JSON object with key "people".
Each person should have: "name", "role", "affiliation".

Text: "Dr. Sarah Chen from MIT presented her research on quantum computing.
Professor James Wilson from Stanford discussed the implications for cryptography.
PhD student Alex Rivera from MIT demonstrated the prototype system."

Return JSON only."""

print("=== MULTI-ENTITY EXTRACTION ===\n")
response = query_llm(prompt, json_mode=True)
parsed = json.loads(response)

# Display as DataFrame
if "people" in parsed:
    people_df = pd.DataFrame(parsed["people"])
    print(people_df.to_string(index=False))
else:
    print(json.dumps(parsed, indent=2))

## Part 3: Pydantic Validation

JSON mode ensures valid JSON, but we also need to validate that the JSON matches our expected *schema* — correct field names, types, and constraints. Pydantic models define the expected structure and automatically validate parsed data.

This is the foundation for production-grade structured extraction: the LLM generates JSON, and Pydantic validates it before the data enters your system.

In [ ]:
if PYDANTIC_AVAILABLE:
    # Define schemas using Pydantic
    class PersonInfo(BaseModel):
        """Schema for extracted person information."""
        name: str = Field(description="Full name of the person")
        age: int = Field(ge=0, le=150, description="Age in years")
        occupation: str = Field(description="Job title or role")
        skills: list[str] = Field(description="List of skills or expertise areas")
        location: str | None = Field(default=None, description="City or country")

    class SentimentResult(BaseModel):
        """Schema for sentiment analysis output."""
        text: str = Field(description="The analyzed text")
        sentiment: str = Field(description="positive, negative, or neutral")
        confidence: float = Field(ge=0.0, le=1.0, description="Confidence score 0-1")
        reasoning: str = Field(description="Brief explanation of the sentiment")

    class EventInfo(BaseModel):
        """Schema for extracted event information."""
        event_name: str = Field(description="Name of the event")
        date: str = Field(description="Date in YYYY-MM-DD format")
        location: str = Field(description="Event location")
        organizer: str | None = Field(default=None, description="Who organized it")
        attendees: int | None = Field(default=None, description="Expected attendance")

    print("Pydantic schemas defined:")
    print(f"  PersonInfo fields:    {list(PersonInfo.model_fields.keys())}")
    print(f"  SentimentResult fields: {list(SentimentResult.model_fields.keys())}")
    print(f"  EventInfo fields:     {list(EventInfo.model_fields.keys())}")
else:
    print("Pydantic not available. Install with: pip install pydantic")

In [ ]:
def extract_with_validation(
    text: str,
    schema_class: type,
    max_retries: int = 2,
) -> Any:
    """Extract structured data with Pydantic validation and retry logic.

    Args:
        text: Input text to extract information from.
        schema_class: Pydantic model class defining the expected schema.
        max_retries: Number of retry attempts for invalid output.

    Returns:
        Validated Pydantic model instance, or None if all retries fail.
    """
    if not PYDANTIC_AVAILABLE or not OLLAMA_AVAILABLE:
        print("Requires Pydantic and Ollama.")
        return None

    # Build the schema description from Pydantic model
    schema_json = json.dumps(schema_class.model_json_schema(), indent=2)

    prompt = f"""Extract information from the following text and return it as JSON
matching this exact schema:

{schema_json}

Text: "{text}"

Return ONLY valid JSON matching the schema above."""

    for attempt in range(max_retries + 1):
        response = query_llm(prompt, json_mode=True)
        try:
            data = json.loads(response)
            result = schema_class.model_validate(data)
            return result
        except (json.JSONDecodeError, ValidationError) as e:
            if attempt < max_retries:
                print(f"  Attempt {attempt + 1} failed, retrying... ({type(e).__name__})")
            else:
                print(f"  All {max_retries + 1} attempts failed: {e}")
                return None

    return None


if PYDANTIC_AVAILABLE and OLLAMA_AVAILABLE:
    # Test extraction with validation
    test_texts = [
        "Alice Johnson, 29, is a machine learning engineer at Google in Mountain View. She specializes in NLP, computer vision, and MLOps.",
        "Bob Lee, aged 45, works as a senior data architect in Seattle. His expertise includes SQL, Spark, Kafka, and cloud infrastructure.",
    ]

    print("=== VALIDATED EXTRACTION ===\n")
    for text in test_texts:
        print(f"Input: {text[:80]}...")
        result = extract_with_validation(text, PersonInfo)
        if result:
            print(f"  Name:       {result.name}")
            print(f"  Age:        {result.age}")
            print(f"  Occupation: {result.occupation}")
            print(f"  Skills:     {result.skills}")
            print(f"  Location:   {result.location}")
        print()
else:
    print("Skipping validation demo (requires Pydantic + Ollama)")

## Part 4: Structured Sentiment Analysis

Let's build a practical pipeline that performs sentiment analysis using a local LLM and returns validated, structured results. This demonstrates how structured output turns a chat model into a reliable classification API.

In [ ]:
def analyze_sentiment(text: str) -> dict[str, Any] | None:
    """Analyze sentiment and return structured result.

    Args:
        text: Text to analyze for sentiment.

    Returns:
        Dictionary with text, sentiment, confidence, and reasoning, or None.
    """
    if not PYDANTIC_AVAILABLE or not OLLAMA_AVAILABLE:
        return None

    prompt = f"""Analyze the sentiment of the following text.
Return JSON with these exact keys:
- "text": the original text (abbreviated to 50 chars max)
- "sentiment": one of "positive", "negative", or "neutral"
- "confidence": a float between 0.0 and 1.0
- "reasoning": a one-sentence explanation

Text: "{text}"

Return ONLY valid JSON."""

    response = query_llm(prompt, json_mode=True)
    try:
        data = json.loads(response)
        result = SentimentResult.model_validate(data)
        return result.model_dump()
    except (json.JSONDecodeError, ValidationError):
        return None


if PYDANTIC_AVAILABLE and OLLAMA_AVAILABLE:
    review_texts = [
        "This product exceeded all my expectations! The build quality is outstanding.",
        "Terrible customer service. I waited 3 hours and nobody helped me.",
        "The hotel was decent. Nothing special but clean and reasonably priced.",
        "I absolutely love this restaurant! Best pasta I've ever had.",
        "The software crashes constantly. Complete waste of money.",
    ]

    print("=== STRUCTURED SENTIMENT ANALYSIS ===\n")
    sentiment_results = []
    for text in review_texts:
        result = analyze_sentiment(text)
        if result:
            sentiment_results.append(result)
            print(f"  [{result['sentiment']:8s}] ({result['confidence']:.2f}) {text[:60]}...")

    if sentiment_results:
        print("\n--- Results Table ---\n")
        sentiment_df = pd.DataFrame(sentiment_results)
        sentiment_df = sentiment_df[["sentiment", "confidence", "text"]]
        print(sentiment_df.to_string(index=False))
else:
    print("Skipping sentiment demo (requires Pydantic + Ollama)")

## Part 5: Function Calling / Tool Use

Function calling is the pattern behind agentic workflows: the LLM decides *which tool to call* and *with what arguments* based on the user's query. We implement this by:

1. Defining available functions with their schemas
2. Asking the LLM to select a function and produce arguments as JSON
3. Executing the selected function
4. Optionally passing the result back to the LLM for a natural language response

This is the same pattern used by MCP (Notebook 06_01) but implemented from scratch to show how it works under the hood.

In [ ]:
# Define available tools / functions
AVAILABLE_TOOLS = {
    "get_weather": {
        "description": "Get the current weather for a city",
        "parameters": {
            "city": "string - the city name",
            "unit": "string - 'celsius' or 'fahrenheit' (default: celsius)",
        },
    },
    "calculate": {
        "description": "Perform a mathematical calculation",
        "parameters": {
            "expression": "string - mathematical expression to evaluate (e.g., '2 + 3 * 4')",
        },
    },
    "search_database": {
        "description": "Search a product database by query",
        "parameters": {
            "query": "string - search query",
            "max_results": "integer - maximum number of results (default: 5)",
        },
    },
}


# Simulated function implementations
def get_weather(city: str, unit: str = "celsius") -> dict[str, Any]:
    """Simulate getting weather for a city.

    Args:
        city: City name.
        unit: Temperature unit (celsius or fahrenheit).

    Returns:
        Dictionary with weather information.
    """
    # Simulated data
    weather_data = {
        "new york": {"temp_c": 22, "condition": "Partly cloudy", "humidity": 65},
        "london": {"temp_c": 15, "condition": "Rainy", "humidity": 80},
        "tokyo": {"temp_c": 28, "condition": "Sunny", "humidity": 55},
        "paris": {"temp_c": 18, "condition": "Overcast", "humidity": 70},
    }
    city_lower = city.lower()
    data = weather_data.get(city_lower, {"temp_c": 20, "condition": "Unknown", "humidity": 50})
    temp = data["temp_c"] if unit == "celsius" else round(data["temp_c"] * 9 / 5 + 32)
    symbol = "C" if unit == "celsius" else "F"
    return {"city": city, "temperature": f"{temp}{symbol}", "condition": data["condition"], "humidity": f"{data['humidity']}%"}


def calculate(expression: str) -> dict[str, Any]:
    """Safely evaluate a mathematical expression.

    Args:
        expression: Math expression string.

    Returns:
        Dictionary with expression and result.
    """
    # Safe evaluation — only allow math operations
    allowed_chars = set("0123456789+-*/.() ")
    if not all(c in allowed_chars for c in expression):
        return {"expression": expression, "result": "Error: invalid characters"}
    try:
        result = eval(expression)  # Safe because we validated chars above
        return {"expression": expression, "result": result}
    except Exception as e:
        return {"expression": expression, "result": f"Error: {e}"}


def search_database(query: str, max_results: int = 5) -> dict[str, Any]:
    """Simulate a product database search.

    Args:
        query: Search query string.
        max_results: Maximum results to return.

    Returns:
        Dictionary with query and matching products.
    """
    products = [
        {"name": "Wireless Mouse", "price": 29.99, "category": "Electronics"},
        {"name": "Mechanical Keyboard", "price": 89.99, "category": "Electronics"},
        {"name": "USB-C Hub", "price": 39.99, "category": "Electronics"},
        {"name": "Monitor Stand", "price": 49.99, "category": "Office"},
        {"name": "Desk Lamp", "price": 34.99, "category": "Office"},
        {"name": "Webcam HD", "price": 59.99, "category": "Electronics"},
        {"name": "Noise Cancelling Headphones", "price": 199.99, "category": "Audio"},
        {"name": "Microphone", "price": 79.99, "category": "Audio"},
    ]
    query_lower = query.lower()
    matches = [p for p in products if query_lower in p["name"].lower() or query_lower in p["category"].lower()]
    return {"query": query, "results": matches[:max_results], "total_found": len(matches)}


TOOL_FUNCTIONS = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_database": search_database,
}

print(f"Defined {len(AVAILABLE_TOOLS)} tools: {list(AVAILABLE_TOOLS.keys())}")

In [ ]:
def function_calling_agent(user_query: str) -> str:
    """Process a user query using function calling.

    The LLM selects which function to call, extracts arguments,
    the function is executed, and the result is returned.

    Args:
        user_query: Natural language user query.

    Returns:
        Final response string incorporating function results.
    """
    if not OLLAMA_AVAILABLE:
        return "Ollama not available."

    # Step 1: Ask LLM to select a tool and extract arguments
    tools_description = json.dumps(AVAILABLE_TOOLS, indent=2)
    selection_prompt = f"""You are an assistant with access to these tools:

{tools_description}

Based on the user's query, select the most appropriate tool and provide the arguments.
Return JSON with exactly these keys:
- "tool": the tool name (one of: {list(AVAILABLE_TOOLS.keys())})
- "arguments": a dictionary of the tool's parameters with values extracted from the query
- "reasoning": one sentence explaining why you chose this tool

User query: "{user_query}"

Return ONLY valid JSON."""

    response = query_llm(selection_prompt, json_mode=True)
    try:
        selection = json.loads(response)
    except json.JSONDecodeError:
        return f"Failed to parse tool selection: {response}"

    tool_name = selection.get("tool", "")
    arguments = selection.get("arguments", {})
    reasoning = selection.get("reasoning", "")

    print(f"  Tool selected: {tool_name}")
    print(f"  Arguments:     {arguments}")
    print(f"  Reasoning:     {reasoning}")

    # Step 2: Execute the selected function
    if tool_name not in TOOL_FUNCTIONS:
        return f"Unknown tool: {tool_name}"

    tool_fn = TOOL_FUNCTIONS[tool_name]
    try:
        result = tool_fn(**arguments)
    except TypeError as e:
        return f"Tool execution failed: {e}"

    print(f"  Result:        {result}")

    # Step 3: Generate natural language response
    response_prompt = f"""The user asked: "{user_query}"

You called the tool '{tool_name}' and got this result:
{json.dumps(result, indent=2)}

Provide a helpful, natural language response to the user based on this result."""

    final_response = query_llm(response_prompt)
    return final_response


if OLLAMA_AVAILABLE:
    test_queries = [
        "What's the weather like in Tokyo?",
        "What is 245 * 18 + 42?",
        "Find me some audio products",
    ]

    print("=== FUNCTION CALLING DEMO ===\n")
    for query in test_queries:
        print(f"User: {query}")
        response = function_calling_agent(query)
        print(f"Assistant: {response}")
        print("-" * 60 + "\n")
else:
    print("Skipping function calling demo (requires Ollama)")

## Part 6: Practical Application — Event Extraction Pipeline

Let's build a complete extraction pipeline that processes text descriptions of events and produces structured, validated records. This demonstrates the full workflow: prompt engineering + JSON mode + Pydantic validation.

In [ ]:
def extract_events(text: str) -> list[dict[str, Any]]:
    """Extract event information from text with validation.

    Args:
        text: Text containing event descriptions.

    Returns:
        List of validated event dictionaries.
    """
    if not PYDANTIC_AVAILABLE or not OLLAMA_AVAILABLE:
        return []

    prompt = f"""Extract all events from the following text.
Return a JSON object with key "events", where each event has:
- "event_name": name of the event
- "date": date in YYYY-MM-DD format (use 2026 if year not specified)
- "location": where it takes place
- "organizer": who organized it (null if unknown)
- "attendees": expected number of attendees (null if unknown)

Text: "{text}"

Return ONLY valid JSON."""

    response = query_llm(prompt, json_mode=True)
    try:
        data = json.loads(response)
        events_raw = data.get("events", [])
    except json.JSONDecodeError:
        return []

    validated_events = []
    for event_raw in events_raw:
        try:
            event = EventInfo.model_validate(event_raw)
            validated_events.append(event.model_dump())
        except ValidationError:
            continue  # Skip invalid entries

    return validated_events


if PYDANTIC_AVAILABLE and OLLAMA_AVAILABLE:
    event_text = """
    The annual Tech Summit will be held on March 15th at the Convention Center in San Francisco,
    organized by TechCorp. They expect around 5000 attendees.

    On April 22nd, the AI Research Workshop takes place at MIT in Cambridge. Dr. Sarah Chen
    is organizing it for approximately 200 researchers.

    The company's quarterly hackathon is scheduled for May 8th at the main office in Austin.
    About 150 engineers will participate.
    """

    print("=== EVENT EXTRACTION PIPELINE ===\n")
    print(f"Input text: {event_text.strip()[:100]}...\n")

    events = extract_events(event_text)
    if events:
        events_df = pd.DataFrame(events)
        print(events_df.to_string(index=False))
        print(f"\nExtracted {len(events)} events successfully.")
    else:
        print("No events extracted.")
else:
    print("Skipping event extraction (requires Pydantic + Ollama)")

## Performance Benchmarking

Let's measure how structured output approaches compare in terms of speed and reliability.

In [ ]:
def benchmark_structured_output(
    num_runs: int = 5,
) -> pd.DataFrame:
    """Benchmark structured output approaches.

    Args:
        num_runs: Number of iterations per approach.

    Returns:
        DataFrame with timing results.
    """
    if not OLLAMA_AVAILABLE:
        print("Ollama required for benchmarking.")
        return pd.DataFrame()

    test_text = "Emma Watson, 35, is a film director in London specializing in drama, documentary, and screenwriting."
    results = []

    # Approach 1: Plain text (no JSON mode)
    prompt_plain = f'Extract name, age, and occupation from: "{test_text}". Return as JSON.'
    times_plain = []
    successes_plain = 0
    for _ in range(num_runs):
        start = time.perf_counter()
        resp = query_llm(prompt_plain, json_mode=False)
        times_plain.append(time.perf_counter() - start)
        try:
            json.loads(resp)
            successes_plain += 1
        except json.JSONDecodeError:
            pass

    results.append({
        "Approach": "Prompt only",
        "Mean (s)": f"{np.mean(times_plain):.2f}",
        "Std (s)": f"{np.std(times_plain):.2f}",
        "JSON Parse Rate": f"{successes_plain}/{num_runs}",
    })

    # Approach 2: JSON mode
    prompt_json = f'Extract name, age, and occupation from: "{test_text}". Return as JSON with keys: name, age, occupation.'
    times_json = []
    successes_json = 0
    for _ in range(num_runs):
        start = time.perf_counter()
        resp = query_llm(prompt_json, json_mode=True)
        times_json.append(time.perf_counter() - start)
        try:
            json.loads(resp)
            successes_json += 1
        except json.JSONDecodeError:
            pass

    results.append({
        "Approach": "JSON mode",
        "Mean (s)": f"{np.mean(times_json):.2f}",
        "Std (s)": f"{np.std(times_json):.2f}",
        "JSON Parse Rate": f"{successes_json}/{num_runs}",
    })

    # Approach 3: JSON mode + Pydantic validation
    if PYDANTIC_AVAILABLE:
        times_validated = []
        successes_validated = 0
        for _ in range(num_runs):
            start = time.perf_counter()
            result = extract_with_validation(test_text, PersonInfo, max_retries=1)
            times_validated.append(time.perf_counter() - start)
            if result is not None:
                successes_validated += 1

        results.append({
            "Approach": "JSON + Pydantic",
            "Mean (s)": f"{np.mean(times_validated):.2f}",
            "Std (s)": f"{np.std(times_validated):.2f}",
            "JSON Parse Rate": f"{successes_validated}/{num_runs}",
        })

    return pd.DataFrame(results)


print("=== STRUCTURED OUTPUT BENCHMARK ===\n")
print("Running benchmarks (this may take 30-60 seconds)...\n")
bench_df = benchmark_structured_output()
if not bench_df.empty:
    print(bench_df.to_string(index=False))
    print("\nNote: JSON mode is slightly slower due to constrained decoding,")
    print("but the reliability improvement is worth the trade-off.")

## Exercises

1. **Custom Schema**: Define a Pydantic model for extracting recipe information (name, ingredients, cooking time, difficulty). Test it on recipe descriptions.

2. **Multi-Turn Function Calling**: Extend the function calling agent to handle follow-up questions (e.g., "What about London?" after asking about Tokyo's weather).

3. **Error Recovery**: Implement more sophisticated retry logic that includes the error message in the retry prompt to help the model correct its output.

4. **Batch Extraction**: Process a list of 10 text descriptions and measure the success rate of schema-validated extraction. Which fields fail most often?

5. **Tool Chaining**: Implement a scenario where the agent needs to call multiple tools sequentially (e.g., search for a product, then calculate tax on its price).

## Key Takeaways

- **JSON mode** (`format="json"` in Ollama) guarantees syntactically valid JSON — use it whenever you need structured output
- **Pydantic validation** adds type safety and constraint checking on top of JSON parsing — essential for production
- **Retry logic** handles the inherent non-determinism of LLM outputs — always plan for occasional failures
- **Function calling** enables agentic tool use by having the LLM select tools and extract arguments as structured JSON
- **Prompt design matters**: clear schema descriptions and examples improve extraction accuracy significantly

## Next Steps

- Review **Notebook 06_01**: MCP Basics for the standardized tool protocol
- Explore [Pydantic Documentation](https://docs.pydantic.dev/) for advanced schema features
- Try [Instructor](https://github.com/jxnl/instructor) for Pydantic-integrated LLM calls

## Resources

- [Ollama JSON Mode](https://ollama.com/blog/openai-compatibility)
- [Pydantic V2 Docs](https://docs.pydantic.dev/latest/)
- [Instructor Library](https://github.com/jxnl/instructor)
- [Function Calling Patterns](https://platform.openai.com/docs/guides/function-calling)
- [Structured Output Best Practices](https://cookbook.openai.com/examples/structured_outputs_intro)